<a href="https://colab.research.google.com/github/Joey-Jireh/eye-of-ra/blob/main/notebooks/week1/week1_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# EYE OF RA — Standard Environment Header
# Paste this as the FIRST cell in every new notebook
# Run it before anything else
# ============================================================

# Install libraries not pre-loaded in Colab
# (runs fast after first time — Colab caches these)
!pip install -q pdfplumber shap xgboost streamlit plotly \
    openpyxl folium streamlit-folium camelot-py

# ============================================================
# Core imports — used across the entire project
# ============================================================

# Data handling
import pandas as pd
import numpy as np

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb

# Explainability
import shap

# PDF extraction
import pdfplumber

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# System
import os
import re
import warnings
warnings.filterwarnings('ignore')

# ============================================================
print("=" * 50)
print("✅ Eye of Ra environment ready")
print("All systems go. Let's catch some corruption. 🇬🇭")
print("=" * 50)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 392.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.5/530.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.2/313.2 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 85.9 MB/s eta 0:00:00
✅ Eye of Ra environment ready
All systems go. Let's catch some corruption. 🇬🇭


In [3]:
# ============================================================
# EYE OF RA — Week 1, Day 2
# Step 1: Verify all our PDF files are here and readable
# ============================================================

import os
import pdfplumber

# List all PDF files we uploaded
pdf_files = [f for f in os.listdir('/content') if f.endswith('.pdf')]
pdf_files.sort()

print(f"✅ Found {len(pdf_files)} PDF files\n")
print("=" * 50)
for f in pdf_files:
    print(f"  📄 {f}")
print("=" * 50)

✅ Found 4 PDF files

  📄 ppa_annual_report_2021.pdf
  📄 ppa_annual_report_2022.pdf
  📄 ppa_annual_report_2023.pdf
  📄 ppa_annual_report_2024.pdf


In [4]:
# ============================================================
# EYE OF RA — Week 1, Day 2
# Cell 2: Verify all PDF files are uploaded and readable
# (with error handling for corrupted uploads)
# ============================================================

pdf_files = [f for f in os.listdir('/content') if f.endswith('.pdf')]
pdf_files.sort()

print(f"Found {len(pdf_files)} PDF files. Checking each one...\n")
print("=" * 50)

good_files = []
bad_files = []

for filename in pdf_files:
    filepath = f'/content/{filename}'
    try:
        with pdfplumber.open(filepath) as pdf:
            num_pages = len(pdf.pages)
            first_page_text = pdf.pages[0].extract_text()

            if first_page_text:
                preview = first_page_text[:100].replace('\n', ' ')
                print(f"✅ {filename}")
                print(f"   Pages: {num_pages}")
                print(f"   Preview: {preview}...")
                good_files.append(filename)
            else:
                print(f"⚠️  {filename}")
                print(f"   Pages: {num_pages}")
                print(f"   WARNING: No readable text on page 1")
                bad_files.append(filename)
    except Exception as e:
        print(f"❌ {filename}")
        print(f"   ERROR: {str(e)[:80]}")
        bad_files.append(filename)
    print()

print("=" * 50)
print(f"✅ Readable files: {len(good_files)}")
print(f"❌ Problem files:  {len(bad_files)}")

if bad_files:
    print("\nThese files need to be re-uploaded:")
    for f in bad_files:
        print(f"  → {f}")

print("=" * 50)

# ------------------------------------------------------------
# COMMIT MESSAGE: "Add cell 2 - PDF verification with error handling"
# ------------------------------------------------------------

Found 4 PDF files. Checking each one...

✅ ppa_annual_report_2021.pdf
   Pages: 78
   Preview: 2021 ANNUAL REPORT Public Procurement Authority...

✅ ppa_annual_report_2022.pdf
   Pages: 72
   Preview: PPA 2022 ANNUAL REPORT PUBLIC PROCUREMENT AUTHORITY...

✅ ppa_annual_report_2023.pdf
   Pages: 88
   Preview: EXECUTIVE SUMMARY Introduction This Annual Report has been prepared in fulfillment of sections 3 (i)...

✅ ppa_annual_report_2024.pdf
   Pages: 86
   Preview: EXECUTIVE SUMMARY Introduction This Annual Report has been prepared in fulfillment of sections 3 (i)...

✅ Readable files: 4
❌ Problem files:  0


In [ ]:
# ============================================================
# EYE OF RA — Week 1, Day 2
# Cell 3: Peek inside one bulletin — understand the structure
# ============================================================

# We'll use the most recent bulletin as our reference
sample_file = '/content/ppa_bulletin_2024_nov_dec.pdf'

print("DEEP INSPECTION: ppa_bulletin_2024_nov_dec.pdf")
print("=" * 60)

with pdfplumber.open(sample_file) as pdf:

    # Look at pages 2, 3, and 4 — page 1 is usually a cover
    for page_num in [1, 2, 3]:
        page = pdf.pages[page_num]
        print(f"\n{'=' * 60}")
        print(f"PAGE {page_num + 1}")
        print(f"{'=' * 60}")

        # Check 1: Is there plain text?
        text = page.extract_text()
        if text:
            print("\n--- RAW TEXT (first 500 characters) ---")
            print(text[:500])
        else:
            print("\n⚠️ No plain text found on this page")

        # Check 2: Are there tables?
        tables = page.extract_tables()
        if tables:
            print(f"\n--- TABLES FOUND: {len(tables)} table(s) on this page ---")
            for i, table in enumerate(tables):
                print(f"\nTable {i+1} — {len(table)} rows x {len(table[0])} columns")
                print("First 3 rows:")
                for row in table[:3]:
                    print(f"  {row}")
        else:
            print("\n⚠️ No tables found on this page")

print("\n" + "=" * 60)
print("✅ Structure inspection complete")
print("=" * 60)

# ------------------------------------------------------------
# COMMIT MESSAGE: "Add cell 3 - bulletin structure inspection"
# ------------------------------------------------------------

DEEP INSPECTION: ppa_bulletin_2024_nov_dec.pdf

PAGE 2

--- RAW TEXT (first 500 characters) ---
Public Procurement Authority: Electronic Bulletin Nov-Dec 2024
e-Bulletin
PPA LAUNCHES THE METHODOLOGY FOR ASSESSING
In this Edition
PROCUREMENT SYSTEMS (MAPS) ASSESSMENT FOR GHANA
 PPA launches the
Methodology for
Assessing
Procurement
Systems (MAPS)
Assessment for
Ghana -Pg. 2,11
 Online
Procurement
Submissions–
Pg. 3, 4,5 & 6
 Editorial - Pg. 8,9
 End of Year
Message from CEO
– Pg.10
On November 26, 2024, the Public Procurement Authority (PPA) in collaboration with the
 Photo Story -
Min

⚠️ No tables found on this page

PAGE 3

--- RAW TEXT (first 500 characters) ---
Public Procurement Authority: Electronic Bulletin Nov-Dec 2024
2024 PROCUREMENT PLAN SUBMISSIONS ON GHANEPS AS AT NOVEMBER 2024
1. Abetifi Presbyterian College Of Education 62. Berekum East Municipal Assembly
2. Ablekuma Central Municipal Assembly 63. Bia East District Assembly
3. Ablekuma North Municipal Assembly 64. B

In [ ]:
# ============================================================
# EYE OF RA — Week 1, Day 2
# Cell 4: Scan ALL pages to find contract award data
# ============================================================

# Keywords that signal contract award data
CONTRACT_KEYWORDS = [
    'contract', 'award', 'tender', 'supplier', 'amount',
    'GH₵', 'GHS', 'cedis', 'bid', 'procur', 'lot'
]

sample_file = '/content/ppa_bulletin_2024_nov_dec.pdf'

print("SCANNING ALL PAGES FOR CONTRACT DATA")
print("=" * 60)

interesting_pages = []

with pdfplumber.open(sample_file) as pdf:
    total_pages = len(pdf.pages)
    print(f"Total pages: {total_pages}\n")

    for page_num in range(total_pages):
        page = pdf.pages[page_num]
        text = page.extract_text()
        tables = page.extract_tables()

        if not text:
            continue

        # Check if this page contains contract-related keywords
        text_lower = text.lower()
        matches = [kw for kw in CONTRACT_KEYWORDS if kw in text_lower]

        if matches:
            interesting_pages.append(page_num)
            print(f"📋 PAGE {page_num + 1} — Keywords found: {matches}")

            # Show first 300 characters of this page
            preview = text[:300].replace('\n', ' ')
            print(f"   Preview: {preview}...")

            # Report tables if any
            if tables:
                print(f"   Tables: {len(tables)} found")
                for i, table in enumerate(tables):
                    if table and table[0]:
                        cols = len(table[0])
                        rows = len(table)
                        print(f"   Table {i+1}: {rows} rows x {cols} cols")
                        print(f"   Headers: {table[0]}")
            print()

print("=" * 60)
print(f"✅ Found {len(interesting_pages)} pages with contract-related content")
print(f"   Pages: {[p+1 for p in interesting_pages]}")
print("=" * 60)

# ----------------------Add cell 4 - full page scan for contract data--------------------------------------
# COMMIT MESSAGE: ""
# ------------------------------------------------------------

SCANNING ALL PAGES FOR CONTRACT DATA
Total pages: 19

📋 PAGE 1 — Keywords found: ['procur']
   Preview: Public Procurement Authority: Electronic Bulletin Nov-Dec 2024 Submit 2024 Procurement Plan Using PPA’s Online Procurement Planning System (http://planning.ppaghana.org/) Page 1...

📋 PAGE 2 — Keywords found: ['procur']
   Preview: Public Procurement Authority: Electronic Bulletin Nov-Dec 2024 e-Bulletin PPA LAUNCHES THE METHODOLOGY FOR ASSESSING In this Edition PROCUREMENT SYSTEMS (MAPS) ASSESSMENT FOR GHANA  PPA launches the Methodology for Assessing Procurement Systems (MAPS) Assessment for Ghana -Pg. 2,11  Online Procure...

📋 PAGE 3 — Keywords found: ['procur']
   Preview: Public Procurement Authority: Electronic Bulletin Nov-Dec 2024 2024 PROCUREMENT PLAN SUBMISSIONS ON GHANEPS AS AT NOVEMBER 2024 1. Abetifi Presbyterian College Of Education 62. Berekum East Municipal Assembly 2. Ablekuma Central Municipal Assembly 63. Bia East District Assembly 3. Ablekuma North Mun...
   Ta

In [ ]:
# ============================================================
# EYE OF RA — Week 1, Day 2
# Cell 5: Deep scan of PPA Annual Report 2024
# ============================================================

CONTRACT_KEYWORDS = [
    'contract', 'award', 'tender', 'supplier', 'amount',
    'GH₵', 'GHS', 'cedis', 'bid', 'value', 'procure',
    'irregularit', 'sanction', 'sole source', 'restricted',
    'single source', 'overpriced', 'entity', 'violation'
]

annual_report = '/content/ppa_annual_report_2024.pdf'

print("DEEP SCAN: PPA Annual Report 2024 (86 pages)")
print("=" * 60)

rich_pages = []

with pdfplumber.open(annual_report) as pdf:
    total_pages = len(pdf.pages)

    for page_num in range(total_pages):
        page = pdf.pages[page_num]
        text = page.extract_text()
        tables = page.extract_tables()

        if not text:
            continue

        text_lower = text.lower()
        matches = [kw for kw in CONTRACT_KEYWORDS if kw in text_lower]

        # Only show pages with 3+ keyword matches — these are the rich ones
        if len(matches) >= 3:
            rich_pages.append(page_num)
            print(f"📋 PAGE {page_num + 1} — Keywords: {matches}")

            preview = text[:400].replace('\n', ' ')
            print(f"   Preview: {preview}...")

            if tables:
                print(f"   Tables: {len(tables)} found")
                for i, table in enumerate(tables):
                    if table and len(table) > 1:
                        print(f"   Table {i+1}: {len(table)} rows x {len(table[0])} cols")
                        print(f"   Headers: {table[0]}")
                        if len(table) > 1:
                            print(f"   Row 2:   {table[1]}")
            print()

print("=" * 60)
print(f"✅ Found {len(rich_pages)} content-rich pages")
print(f"   Pages: {[p+1 for p in rich_pages]}")
print("=" * 60)

# ------------------------------------------------------------
# COMMIT MESSAGE: "Add cell 5 - annual report deep scan"
# ------------------------------------------------------------

DEEP SCAN: PPA Annual Report 2024 (86 pages)
📋 PAGE 3 — Keywords: ['contract', 'award', 'tender', 'procure']
   Preview: Summary (Weighted) 40 36.85 34.5 35 30 25 e 18.62 r o 20 16.56 c S 15 10.52 10.52 8.2 9.08 10 5 0 Management Systems Information & Procurement Process Contract Management Communication KPCs 2022 2023 Under the MANAGEMENT SYSTEMS, the score for 2023 was 70.11% compared to 54.65% in 2022. Management provided the necessary support to the conduct of procurement in the Entities. Evidence gathered also ...
   Tables: 1 found
   Table 1: 10 rows x 15 cols
   Headers: [None, None, None, None, None, None, None, None, None, None, '', None, None, None, None]
   Row 2:   [None, None, None, None, None, None, None, None, '', '', '', None, None, None, None]

📋 PAGE 4 — Keywords: ['contract', 'award', 'tender', 'value', 'procure', 'restricted', 'single source']
   Preview: TABLE A: PERFORMANCE MEASUREMENT INDICATORS FOR 2023 RESULTS INDICATOR METRICS CRITERIA 2023 2022 a) % of open 

In [5]:
# ============================================================
# EYE OF RA — Week 1, Day 2
# Cell 6: Extract the ratification tables from Annual Report
# These are our gold — labelled procurement transactions
# ============================================================

annual_report = '/content/ppa_annual_report_2024.pdf'

# Pages 33-37 contain the ratification request tables
# Let's extract ALL rows from those tables
TARGET_PAGES = [32, 33, 34, 35, 36, 37]  # 0-indexed so page 33 = index 32

print("EXTRACTING RATIFICATION TABLES — Pages 33–38")
print("=" * 60)

all_ratification_rows = []

with pdfplumber.open(annual_report) as pdf:
    for page_num in TARGET_PAGES:
        page = pdf.pages[page_num]
        tables = page.extract_tables()

        print(f"\nPAGE {page_num + 1}:")

        if not tables:
            print("  No tables found")
            continue

        for table in tables:
            # Only process tables with 4+ columns
            # (our ratification table has 5 columns)
            if not table or len(table[0]) < 4:
                continue

            print(f"  Table: {len(table)} rows x {len(table[0])} cols")
            print(f"  Headers: {table[0]}")
            print()

            # Extract every row
            for row in table[1:]:  # skip header row
                if row and any(cell for cell in row if cell):
                    all_ratification_rows.append(row)
                    print(f"  ROW: {row}")

print("\n" + "=" * 60)
print(f"✅ Total rows extracted: {len(all_ratification_rows)}")
print("=" * 60)

# ------------------------------------------------------------
# COMMIT MESSAGE: "Add cell 6 - extract ratification tables from annual report"
# ------------------------------------------------------------

EXTRACTING RATIFICATION TABLES — Pages 33–38

PAGE 33:
  Table: 5 rows x 6 cols
  Headers: ['', 'Quarter 1', 'Quarter 2', 'Quarter 3', 'Quarter 4', 'Total']

  ROW: ['Received', '78', '104', '77', '92', '351']
  ROW: ['Approved', '36', '89', '73', '68', '266']
  ROW: ['Rejected', '1', '1', '0', '0', '2']
  ROW: ['Queried', '6', '13', '4', '8', '31']
  Table: 7 rows x 5 cols
  Headers: ['N0', 'ENTITY', 'REQUEST', 'INVESTIGAT\nION Report', 'DECISI\nON']

  ROW: ['1.', 'Coastal Development\nAuthority (CODA)', 'Engaged Messrs. TFS Ventures for the\nsupply of rice.', 'Recommended', 'Ratified']
  ROW: ['2.', 'National Biosafety\nAuthority (NBA)', 'Engaged Messrs. FRENGLISH Wave\nConsult for the procurement infraction for\nthe payment of translation', 'Recommended', 'Ratified']
  ROW: ['3.', 'Ministry of Finance (MoF)', 'Request for Ratification to procure\nactivities for delivering of programmes for\nNational Entrepreneurship and Innovation\nProgramme (NEIP)', 'Recommended', 'Ratified']
  RO

In [6]:
# ============================================================
# EYE OF RA — Week 1, Day 2
# Cell 7: Clean and structure the ratification data
# Turn raw extracted rows into a proper dataset
# ============================================================

import pandas as pd
import re

# Raw data from our extraction
# We'll rebuild it cleanly here
raw_data = [
    # Page 33
    ['1', 'Coastal Development Authority (CODA)', 'Engaged Messrs. TFS Ventures for the supply of rice.', 'Recommended', 'Ratified'],
    ['2', 'National Biosafety Authority (NBA)', 'Engaged Messrs. FRENGLISH Wave Consult for procurement infraction for payment of translation', 'Recommended', 'Ratified'],
    ['3', 'Ministry of Finance (MoF)', 'Request for Ratification to procure activities for delivering of programmes for National Entrepreneurship and Innovation Programme (NEIP)', 'Recommended', 'Ratified'],
    ['4', 'Ghana Investment Fund for Electronic Communications (GIFEC)', 'Engaged the services of Messrs Ascend Digital Solutions Limited for the Provisions of Project management services for the year 2023.', 'Recommended', 'Ratified'],
    ['5', 'Ghana Investment Fund for Electronic Communications (GIFEC)', 'Ratification for the continuous engagement of Messrs. Ascend Digital Solutions Limited for the provision of Warehouse Management Service for the year 2023', 'Recommended', 'Ratified'],
    ['6', 'University of Ghana (UG)', 'Request for ratification for the procurement of some Medical and Diagnostics Services on behalf of staff and students from the University of Ghana Medical Centre Limited (UGMC)', 'Recommended', 'Ratified'],
    # Page 34
    ['7', 'Ghana National Gas Company (Ghana Gas)', 'Request for ratification of the Variation of the Fabrication and Replacement of severely corroded 4No. Debutanizer Reflux Condenser Structures at the Gas Processing Plant at an additional cost', 'Recommended', 'Ratified'],
    ['8', 'Ghana National Gas Company (Ghana Gas)', 'Engaged Messrs. Agenda Commercial Limited for the procurement of four 4No. Uninterruptible Power Supply (UPS)', 'Recommended', 'Ratified'],
    ['9', 'Accra Technical University (ATU)', 'Engaged Mrs. Veronica Quarno, the Landlord at St. Michael Townhouse and Apartment Conference of the Magistrates and Judges Association', 'Recommended', 'Ratified'],
    ['10', 'Greater Accra Regional Hospital (Ridge Hospital)', 'Engaged Messrs. Cummins Ghana Limited to service two 25000KVA Gensets', 'Recommended', 'Ratified'],
    ['11', 'Bulk Oil Storage Transport (BOST)', 'Request for ratification for the Conveyance and Installation of 400KVA & 1000KVA Generator Sets to and from Bolgatanga to Kumasi', 'Recommended', 'Ratified'],
    ['12', 'Bulk Oil Storage Transport (BOST)', 'Engaged messrs Tringa Oil to serve as an Official Sales Agent for the sale of Petroleum in Mali for a period of two years', 'Recommended', 'Not Ratified'],
    ['13', 'Ministry of Transport (MoT)', 'Engaged Messrs. Amaris Terminal Ghana Limited for the provision of transport services for the evacuation of foodstuffs', 'Recommended', 'Ratified'],
    ['14', 'Ministry of Education (MoE)', 'Engaged Messrs. Essydel Event for the provision of event management services for the launch of the Accra World Book Capital 2023', 'Recommended', 'Ratified'],
    ['15', 'Social Security and National Insurance Trust (SSNIT)', 'Engaged Messrs. Infonaligy Limited for the provision of Local Support for Oracle Software Application', 'Recommended', 'Ratified'],
    ['16', 'Electoral Commission (EC)', 'Request to ratify the contract extension to Messrs. Hygeena Limited for the provision of cleaning services to the New Electoral Commission Office Building from February 2022 to February 2024', 'Recommended', 'Ratified'],
    ['17', 'National Identification Authority (NIA)', 'Engaged Messrs. State Insurance Company Limited for the provision of Insurance Services.', 'Recommended', 'Ratified'],
    # Page 35
    ['18', 'Bulk Oil Storage and Transportation Company Limited (BOST)', 'Engaged the services of Messrs. Yin Namal Limited for the Expansion of Fuel Quality Control Laboratory Building at Kumasi depot', 'Recommended', 'Not Ratified'],
    ['19', 'Pharmacy Council', 'Engaged Messrs. RX Health Information Systems Limited to develop and implement the National Electronic Pharmacy Platform (NEPP)', 'Recommended', 'Ratified'],
    ['20', 'Ghana Commodity Exchange (GCX)', 'Engaged Messrs. DESS Inc. for the provision of the Exchange Commodity Trading Platform from November 2018 to March 2021', 'Recommended', 'Ratified'],
    ['21', 'Ghana Investment Fund for Electronic Communication (GIFEC)', 'Engaged messrs CSS Precise Systems Limited for the maintenance and operation of their Satellite Hub Services from 9th May 2023 to 8th May 2024', 'Recommended', 'Ratified'],
    ['22', 'Ministry of Finance (MoF)', 'Engaged Messrs AB Namas Ghana Limited for the refurbishment and rehabilitation of bungalow No. C6 (11) Djebobo Street, Redco-Madina', 'Recommended', 'Not Ratified'],
    ['23', 'Minerals Income Investment Fund (MIIF)', 'Engaged messrs IMARA Corporate Finance SA to provide transaction advisory services for the monetization of gold royalties (Project Kingdom).', 'Recommended', 'Not Ratified'],
    ['24', 'Centre for Plant Medicine Research', 'Requested for approval of their new policy for the procurement of perishable and nonperishable plant raw materials', 'Recommended', 'Ratified'],
    ['25', 'Kumasi Metropolitan Assembly (KMA)', 'Requested for ratification of procurement infraction.', 'Recommended', 'Ratified'],
    ['26', 'Ministry of Youth and Sports (MoYS)', 'Engaged Messrs. Fadhab Investment Company for the Provision of Travel and Tour Services to Airlift Transport Accommodate Officials and Supporters for the African Cup of Nations in Cote D Ivoire at an additional cost', 'Recommended', 'Ratified'],
    ['27', 'National Blood Service Ghana (NBSG)', 'Engaged Messrs. Investrade International Co. Ltd for the supply of Blood Collection Bags', 'Recommended', 'Ratified'],
    ['28', 'National Petroleum Authority (NPA)', 'Ratification for extending the contract of Saeed Sasco Investment Limited for the management of NPAs Bulk Road Vehicle Tanker Parking Lot and Office Building at Kpone from December 2021 to December 2022', 'Recommended', 'Ratified'],
    # Page 36
    ['29', 'Ghana Revenue Authority (GRA)', 'Engaged Messrs Digitalgov Limited to host the Software Infrastructure and Services for E-Levy transaction from May 2023 to 31st July 2024', 'Recommended', 'Ratified'],
    ['30', 'Ministry of Defence (MoD)', 'Request to ratify the installation of early fire warning systems', 'Recommended', 'Ratified'],
    ['31', 'Ministry of Information', 'Requested for the ratification for the extension to telecast live developmental project to include more participation of the local media in the Districts across the 16 regions at an additional cost.', 'Recommended', 'Ratified'],
    ['32', 'Ghana School of Law (GSL)', 'Request to ratify various procurement activities undertaken by the Ghana School of Law the General Legal Council and the Independent Examinations Committee', 'Recommended', 'Ratified'],
    ['33', 'Ministry of Local Government Decentralization and Rural Development (MLGDRD)', 'Request for the ratification of the variation of contract awarded to Messrs. Job 2k Limited for the Erection and Completion of 2No. Senior Bungalows at Atebubu in the Bono East', 'Recommended', 'Ratified'],
    ['34', 'Ministry of Youth and Sports (MoYS)', 'Engaged various Service providers for the provision of various Technical Services during the 13th African Games.', 'Recommended', 'Ratified'],
    ['35', 'Ministry of Local Government Decentralization and Rural Development (MLGDRD)', 'Request for ratification for the Variation of Contract awarded to Messrs. Wisdom Wallet Company Limited for the erection and Completion of 2No. Senior Staff Bungalows at Kukuom in the Ahafo Region.', 'Recommended', 'Ratified'],
    ['36', 'Ministry of Local Government Decentralization and Rural Development (MLGDRD)', 'Request for ratification for the Variation of Contract awarded to Messrs. Dagbene Borns Limited for the Erection and completion of 1No. 3-Storey Administration Block for Savannah Regional Coordinating Council at Damongo in the Savannah Region.', 'Recommended', 'Ratified'],
    # Page 37
    ['37', 'Ministry of Youth and Sports (MoYS)', 'Engaged Messrs. Inpath technologies Ghana Limited for the supply and provision of technological tools and Software for the Core Operation of the 13th African Games.', 'Recommended', 'Ratified'],
    ['38', 'Ghana Export Promotion Authority (GEPA)', 'Engaged the Services of Messrs. Hotel Investments Ghana Limited (Labadi Beach Hotel) for hosting the Ghana Rwanda Business Forum.', 'Recommended', 'Ratified'],
    ['39', 'Ghana Export Promotion Authority (GEPA)', 'Engaged Messrs Human Capital International for a four-day workshop from 17th to 25th June 2022 for Middle Management.', 'Recommended', 'Not Ratified'],
    ['40', 'Ministry of Local Government Decentralization and Rural Development (MLGDRD)', 'Engaged Messrs. HB Bekam Ghana Limited for the Erection and Completion of 1No. 2-Storey Administration Block for the Department of Ghana Health Services at Kintampo in the Bono East Region.', 'Recommended', 'Ratified'],
    ['41', 'Ministry of Local Government Decentralization and Rural Development (MLGDRD)', 'Engaged Messrs. Rock Impex Limited for the ratification of variation of contract for the Erection and Completion of 3No. Senior Bungalows at Nalerigu in the North East region.', 'Recommended', 'Ratified'],
    ['42', 'Ghana Export Promotion Authority (GEPA)', 'Engaged Messrs. Manage Advertising for the provision of Marketing and Promotional Services for the Aburi Art and Craft Village Official Opening and Exhibition Event.', 'Recommended', 'Ratified'],
    ['43', 'Ghana Export Promotion Authority (GEPA)', 'Engaged Messrs. Insight Advertising Limited for the provision of Media Services for Aburi Art and Craft Village Official Opening and Exhibition Event.', 'Recommended', 'Ratified'],
    ['44', 'Ghana Maritime Authority (GMA)', 'Engaged Messrs. Kaysens Gaisie Ltd. and Trustlink Ventures Ltd for the Procurement of Extra Various Food Items and Beverages for Xmas.', 'Recommended', 'Ratified'],
    ['45', 'Ghana Geological Survey Authority (GGSA)', 'Engaged Messrs. Forte Logistics for the Supply of Drilling Rig Parts and Accessories', 'Recommended', 'Ratified'],
]

# ============================================================
# Build the dataframe
# ============================================================

df = pd.DataFrame(raw_data, columns=[
    'contract_id',
    'entity',
    'contract_description',
    'investigation_result',
    'decision'
])

# ============================================================
# Add engineered columns
# ============================================================

# Year source
df['year'] = 2024

# Binary label — this is what our model will predict
# 1 = flagged/problematic (Not Ratified)
# 0 = clean (Ratified)
df['fraud_label'] = df['decision'].apply(
    lambda x: 1 if 'Not' in str(x) else 0
)

# Extract supplier name from description
# Most descriptions follow "Engaged Messrs. [SUPPLIER] for..."
def extract_supplier(text):
    # Look for "Messrs." pattern
    match = re.search(r'Messrs\.?\s+([A-Z][^f][^\n]+?)(?:\s+for|\s+to\s)', text)
    if match:
        return match.group(1).strip()
    return 'Unknown'

df['supplier'] = df['contract_description'].apply(extract_supplier)

# Detect contract variations (cost escalations — a key fraud signal)
df['is_variation'] = df['contract_description'].str.lower().apply(
    lambda x: 1 if any(word in x for word in
    ['variation', 'additional cost', 'extension', 'extending']) else 0
)

# Detect repeat entity appearances (concentration risk)
entity_counts = df['entity'].value_counts()
df['entity_frequency'] = df['entity'].map(entity_counts)

# Detect sole/single source indicators
df['is_sole_source'] = df['contract_description'].str.lower().apply(
    lambda x: 1 if any(word in x for word in
    ['single source', 'sole source', 'ratification',
     'infraction', 'without tender']) else 0
)

# ============================================================
# Preview the dataset
# ============================================================

print("EYE OF RA — RATIFICATION DATASET")
print("=" * 60)
print(f"Total contracts: {len(df)}")
print(f"Fraud labels (Not Ratified): {df['fraud_label'].sum()}")
print(f"Clean labels (Ratified): {(df['fraud_label'] == 0).sum()}")
print(f"Contract variations detected: {df['is_variation'].sum()}")
print(f"Sole source indicators: {df['is_sole_source'].sum()}")
print()
print("FRAUD LABEL DISTRIBUTION:")
print(df['decision'].value_counts())
print()
print("ENTITIES WITH MULTIPLE CONTRACTS (concentration risk):")
print(entity_counts[entity_counts > 1])
print()
print("SAMPLE ROWS WITH ALL FEATURES:")
print(df[['contract_id', 'entity', 'supplier',
          'fraud_label', 'is_variation',
          'entity_frequency', 'is_sole_source']].head(10).to_string())
print()
print("=" * 60)
print("✅ Dataset structured successfully")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print("=" * 60)

# ------------------------------------------------------------
# COMMIT MESSAGE: "Add cell 7 - clean and structure ratification dataset"
# ------------------------------------------------------------

EYE OF RA — RATIFICATION DATASET
Total contracts: 45
Fraud labels (Not Ratified): 5
Clean labels (Ratified): 40
Contract variations detected: 9
Sole source indicators: 13

FRAUD LABEL DISTRIBUTION:
decision
Ratified        40
Not Ratified     5
Name: count, dtype: int64

ENTITIES WITH MULTIPLE CONTRACTS (concentration risk):
entity
Ministry of Local Government Decentralization and Rural Development (MLGDRD)    5
Ghana Export Promotion Authority (GEPA)                                         4
Ministry of Youth and Sports (MoYS)                                             3
Ghana Investment Fund for Electronic Communications (GIFEC)                     2
Ghana National Gas Company (Ghana Gas)                                          2
Ministry of Finance (MoF)                                                       2
Bulk Oil Storage Transport (BOST)                                               2
Name: count, dtype: int64

SAMPLE ROWS WITH ALL FEATURES:
  contract_id                     

In [7]:
# ============================================================
# EYE OF RA — Week 1, Day 2
# Cell 9: Extract ratification tables from ALL annual reports
# 2021, 2022, 2023, 2024 — one master dataset
# ============================================================

import pandas as pd
import pdfplumber
import re
import os

# ============================================================
# Configuration — our four annual reports
# ============================================================

annual_reports = [
    {'file': '/content/ppa_annual_report_2021.pdf', 'year': 2021},
    {'file': '/content/ppa_annual_report_2022.pdf', 'year': 2022},
    {'file': '/content/ppa_annual_report_2023.pdf', 'year': 2023},
    {'file': '/content/ppa_annual_report_2024.pdf', 'year': 2024},
]

# ============================================================
# Helper functions
# ============================================================

def extract_supplier(text):
    """Extract supplier name from contract description"""
    match = re.search(
        r'Messrs\.?\s+([A-Z][^f][^\n]+?)(?:\s+for|\s+to\s|\s+in\s)',
        str(text)
    )
    if match:
        return match.group(1).strip()
    return 'Unknown'

def is_ratification_table(table):
    """Check if a table is a ratification request table"""
    if not table or not table[0]:
        return False
    headers = [str(h).lower() for h in table[0] if h]
    header_text = ' '.join(headers)
    return (
        ('entity' in header_text or 'n0' in header_text or 'no' in header_text)
        and ('request' in header_text or 'decision' in header_text)
        and len(table[0]) >= 4
    )

def extract_ratification_rows(filepath, year):
    """Extract all ratification rows from one annual report"""
    rows = []

    try:
        with pdfplumber.open(filepath) as pdf:
            total_pages = len(pdf.pages)
            print(f"\n  Scanning {total_pages} pages...")

            for page_num in range(total_pages):
                page = pdf.pages[page_num]
                tables = page.extract_tables()

                if not tables:
                    continue

                for table in tables:
                    if not is_ratification_table(table):
                        continue

                    # Found a ratification table — extract rows
                    for row in table[1:]:  # skip header
                        if not row or not any(cell for cell in row if cell):
                            continue

                        # Clean each cell
                        cleaned = [
                            str(cell).replace('\n', ' ').strip()
                            if cell else ''
                            for cell in row
                        ]

                        # Skip rows with no meaningful content
                        if not cleaned[1] and not cleaned[2]:
                            continue

                        # Map to standard columns
                        # Handle both 5-col and 4-col table formats
                        if len(cleaned) >= 5:
                            rows.append({
                                'year': year,
                                'contract_id': f"{year}_{cleaned[0]}",
                                'entity': cleaned[1],
                                'contract_description': cleaned[2],
                                'investigation_result': cleaned[3],
                                'decision': cleaned[4],
                            })
                        elif len(cleaned) == 4:
                            rows.append({
                                'year': year,
                                'contract_id': f"{year}_{cleaned[0]}",
                                'entity': cleaned[1],
                                'contract_description': cleaned[2],
                                'investigation_result': '',
                                'decision': cleaned[3],
                            })

    except Exception as e:
        print(f"  ❌ Error processing {filepath}: {e}")

    return rows

# ============================================================
# Extract from all four reports
# ============================================================

all_rows = []

for report in annual_reports:
    year = report['year']
    filepath = report['file']

    if not os.path.exists(filepath):
        print(f"⚠️  File not found: {filepath}")
        continue

    print(f"\n{'=' * 50}")
    print(f"Processing {year} Annual Report...")
    rows = extract_ratification_rows(filepath, year)
    all_rows.extend(rows)
    print(f"  ✅ Extracted {len(rows)} rows from {year}")

# ============================================================
# Build master dataframe
# ============================================================

print(f"\n{'=' * 50}")
print("BUILDING MASTER DATASET...")
print(f"{'=' * 50}")

df_master = pd.DataFrame(all_rows)

if df_master.empty:
    print("⚠️  No data extracted. Check file names match exactly.")
else:
    # Clean decision column
    df_master['decision'] = df_master['decision'].str.strip()

    # Binary fraud label
    df_master['fraud_label'] = df_master['decision'].apply(
        lambda x: 1 if 'Not' in str(x) else 0
    )

    # Feature engineering
    df_master['supplier'] = df_master['contract_description'].apply(
        extract_supplier
    )

    df_master['is_variation'] = df_master['contract_description'].str.lower().apply(
        lambda x: 1 if any(w in x for w in
        ['variation', 'additional cost', 'extension',
         'extending', 'escalat']) else 0
    )

    df_master['is_sole_source'] = df_master['contract_description'].str.lower().apply(
        lambda x: 1 if any(w in x for w in
        ['single source', 'sole source', 'ratification',
         'infraction', 'without tender']) else 0
    )

    entity_counts = df_master['entity'].value_counts()
    df_master['entity_frequency'] = df_master['entity'].map(entity_counts)

    # ============================================================
    # Summary report
    # ============================================================

    print(f"\n✅ MASTER DATASET READY")
    print(f"{'=' * 50}")
    print(f"Total contracts:          {len(df_master)}")
    print(f"Years covered:            {sorted(df_master['year'].unique())}")
    print(f"Unique entities:          {df_master['entity'].nunique()}")
    print(f"Fraud labels (flagged):   {df_master['fraud_label'].sum()}")
    print(f"Clean labels:             {(df_master['fraud_label']==0).sum()}")
    print(f"Contract variations:      {df_master['is_variation'].sum()}")
    print(f"Sole source indicators:   {df_master['is_sole_source'].sum()}")
    print(f"Fraud rate:               {df_master['fraud_label'].mean():.1%}")
    print()

    print("CONTRACTS PER YEAR:")
    print(df_master['year'].value_counts().sort_index())
    print()

    print("FRAUD LABELS PER YEAR:")
    print(df_master.groupby('year')['fraud_label'].sum())
    print()

    print("TOP 10 ENTITIES BY CONTRACT FREQUENCY:")
    print(entity_counts.head(10))
    print()

    print("SAMPLE ROWS:")
    print(df_master[['contract_id', 'entity', 'fraud_label',
                      'is_variation', 'is_sole_source',
                      'entity_frequency']].head(10).to_string())

    print(f"\n{'=' * 50}")
    print(f"Shape: {df_master.shape[0]} rows x {df_master.shape[1]} columns")
    print(f"{'=' * 50}")

# ------------------------------------------------------------
# COMMIT MESSAGE: "Add cell 9 - master dataset from all annual reports"
# ------------------------------------------------------------


Processing 2021 Annual Report...

  Scanning 78 pages...
  ✅ Extracted 70 rows from 2021

Processing 2022 Annual Report...

  Scanning 72 pages...
  ✅ Extracted 68 rows from 2022

Processing 2023 Annual Report...

  Scanning 88 pages...
  ✅ Extracted 83 rows from 2023

Processing 2024 Annual Report...

  Scanning 86 pages...
  ✅ Extracted 46 rows from 2024

BUILDING MASTER DATASET...

✅ MASTER DATASET READY
Total contracts:          267
Years covered:            [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Unique entities:          145
Fraud labels (flagged):   23
Clean labels:             244
Contract variations:      26
Sole source indicators:   47
Fraud rate:               8.6%

CONTRACTS PER YEAR:
year
2021    70
2022    68
2023    83
2024    46
Name: count, dtype: int64

FRAUD LABELS PER YEAR:
year
2021     1
2022     7
2023    10
2024     5
Name: fraud_label, dtype: int64

TOP 10 ENTITIES BY CONTRACT FREQUENCY:
entity
National Petroleum Authority (NPA)       

In [8]:
# ============================================================
# EYE OF RA — Week 1, Day 2
# Cell 10: Clean master dataset and save to CSV
# ============================================================

import pandas as pd
import re
import os

# ============================================================
# Step 1: Remove empty/ghost rows
# ============================================================

initial_count = len(df_master)

df_clean = df_master[
    df_master['entity'].str.strip() != ''
].copy()

df_clean = df_clean[
    df_clean['contract_description'].str.strip() != ''
].copy()

df_clean = df_clean.reset_index(drop=True)

print(f"Removed {initial_count - len(df_clean)} ghost rows")
print(f"Remaining: {len(df_clean)} contracts")

# ============================================================
# Step 2: Standardise entity names
# Entity name mapping — consolidate variations
# ============================================================

entity_mapping = {
    # BOST variations
    'BOST': 'Bulk Oil Storage and Transportation Company Limited (BOST)',
    'Bulk Oil Storage Transport (BOST)': 'Bulk Oil Storage and Transportation Company Limited (BOST)',
    'Bulk Oil & Storage Transport (BOST)': 'Bulk Oil Storage and Transportation Company Limited (BOST)',

    # Ministry of Finance variations
    'Ministry of Finance': 'Ministry of Finance (MoF)',
    'Ministry of Finance (MOF)': 'Ministry of Finance (MoF)',

    # University of Ghana variations
    'University of Ghana': 'University of Ghana (UG)',
    'University of Ghana National (UG)': 'University of Ghana (UG)',

    # SSNIT variations
    'SSNIT': 'Social Security and National Insurance Trust (SSNIT)',

    # NIA variations
    'National Identification Authority': 'National Identification Authority (NIA)',

    # GRA variations
    'Ghana Revenue Authority': 'Ghana Revenue Authority (GRA)',

    # GIFEC variations
    'Ghana Investment Fund for Electronic Communication (GIFEC)':
        'Ghana Investment Fund for Electronic Communications (GIFEC)',

    # MLGDRD variations
    'Ministry of Local Government, Decentralization and Rural Development':
        'Ministry of Local Government Decentralization and Rural Development (MLGDRD)',
    'Ministry of Local Government':
        'Ministry of Local Government Decentralization and Rural Development (MLGDRD)',

    # MoYS variations
    'Ministry of Youth and Sports': 'Ministry of Youth and Sports (MoYS)',

    # Korle Bu variations
    'Korle Bu Teaching Hospital': 'Korle Bu Teaching Hospital (KBTH)',
    'Korle-Bu Teaching Hospital (KBTH)': 'Korle Bu Teaching Hospital (KBTH)',
}

df_clean['entity'] = df_clean['entity'].replace(entity_mapping)

# ============================================================
# Step 3: Recompute entity frequency with clean names
# ============================================================

entity_counts_clean = df_clean['entity'].value_counts()
df_clean['entity_frequency'] = df_clean['entity'].map(entity_counts_clean)

# ============================================================
# Step 4: Add repeat supplier detection
# Same supplier appearing across multiple contracts
# ============================================================

supplier_counts = df_clean['supplier'].value_counts()
df_clean['supplier_frequency'] = df_clean['supplier'].map(supplier_counts)

# Flag suppliers appearing 2+ times (concentration risk)
df_clean['repeat_supplier'] = (df_clean['supplier_frequency'] > 1).astype(int)

# ============================================================
# Step 5: Save to CSV
# ============================================================

os.makedirs('/content/data', exist_ok=True)
output_path = '/content/data/eye_of_ra_master_dataset.csv'
df_clean.to_csv(output_path, index=False)

# ============================================================
# Step 6: Final summary
# ============================================================

print(f"\n{'=' * 55}")
print("✅ MASTER DATASET CLEANED AND SAVED")
print(f"{'=' * 55}")
print(f"Total contracts:           {len(df_clean)}")
print(f"Years covered:             {sorted(df_clean['year'].unique())}")
print(f"Unique entities:           {df_clean['entity'].nunique()}")
print(f"Unique suppliers:          {df_clean['supplier'].nunique()}")
print(f"Fraud labels (flagged):    {df_clean['fraud_label'].sum()}")
print(f"Clean labels:              {(df_clean['fraud_label']==0).sum()}")
print(f"Fraud rate:                {df_clean['fraud_label'].mean():.1%}")
print(f"Contract variations:       {df_clean['is_variation'].sum()}")
print(f"Sole source indicators:    {df_clean['is_sole_source'].sum()}")
print(f"Repeat suppliers:          {df_clean['repeat_supplier'].sum()}")
print()
print("TOP 10 ENTITIES (clean names, by frequency):")
print(entity_counts_clean.head(10))
print()
print("FRAUD LABELS PER YEAR:")
print(df_clean.groupby('year')['fraud_label'].sum())
print()
print("REPEAT SUPPLIERS (appearing 3+ times):")
print(supplier_counts[supplier_counts >= 3])
print()
print(f"Saved to: {output_path}")
print(f"Shape: {df_clean.shape[0]} rows x {df_clean.shape[1]} columns")
print(f"{'=' * 55}")

# ------------------------------------------------------------
# COMMIT MESSAGE: "Add cell 10 - clean dataset, fix entity names, save CSV"
# ------------------------------------------------------------

Removed 5 ghost rows
Remaining: 262 contracts

✅ MASTER DATASET CLEANED AND SAVED
Total contracts:           262
Years covered:             [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Unique entities:           133
Unique suppliers:          119
Fraud labels (flagged):    23
Clean labels:              239
Fraud rate:                8.8%
Contract variations:       26
Sole source indicators:    47
Repeat suppliers:          150

TOP 10 ENTITIES (clean names, by frequency):
entity
Bulk Oil Storage and Transportation Company Limited (BOST)                       15
Ministry of Finance (MoF)                                                        13
National Petroleum Authority (NPA)                                               12
Social Security and National Insurance Trust (SSNIT)                             10
University of Ghana (UG)                                                          9
Korle Bu Teaching Hospital (KBTH)                                           

In [9]:
# Cell 11 — Reload master dataset
df = pd.read_csv('/content/data/eye_of_ra_master_dataset.csv')
print(f"✅ Master dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Fraud labels: {df['fraud_label'].sum()} | Clean: {(df['fraud_label'] == 0).sum()}")
df.head()

# Git: loaded master dataset for Day 3 work

✅ Master dataset loaded: 262 rows x 13 columns
Fraud labels: 23 | Clean: 239


,year,contract_id,entity,contract_description,investigation_result,decision,fraud_label,supplier,is_variation,is_sole_source,entity_frequency,supplier_frequency,repeat_supplier
0,2021,2021_1.,Ministry of Finance (MoF),"Engaged Omnia Strategy LLP, White and Case LLP...",Recommended,Ratified,0,Unknown,0,0,13,138,1
1,2021,2021_2.,National Petroleum Authority (NPA),"Engaged Mamphey Developers Limited, a Consulta...",Recommended,Ratified,0,Unknown,0,0,12,138,1
2,2021,2021_3.,National Identification Authority (NIA),Engaged Messrs Activate Africa to provide cons...,Recommended,Ratified,0,Activate Africa,0,0,5,2,1
3,2021,2021_4.,Ghana National Petroleum Corporation (GNPC),Engaged Mr. David Sharp as an expert Witness f...,Recommended,Ratified,0,Unknown,0,0,3,138,1
4,2021,2021_5.,Bank of Ghana,Procurement of COVID-19 Emergency purchases fo...,Recommended,Ratified,0,Unknown,0,0,5,138,1


In [11]:
# Cell 12 — Extract flagged entities from Auditor-General reports

ag_files = [
    '/content/audit_report_public_board_2021.pdf',
    '/content/audit_report_ministry_2021.pdf',
    '/content/audit_report_public_board_2022.pdf',
    '/content/audit_report_ministry_2022.pdf',
    '/content/audit_report_public_board_2023.pdf',
    '/content/audit_report_ministry_2023.pdf',
]

# Keywords that signal an irregularity finding
irregularity_keywords = [
    'irregularit', 'procurement', 'sole source', 'single source',
    'without competitive', 'inflated', 'overpriced', 'ghost',
    'unretired', 'unsupported', 'misapplication', 'not ratified',
    'variance', 'overpayment', 'duplicate'
]

flagged_entities = []

for filepath in ag_files:
    year = filepath.split('_')[-1].replace('.pdf', '')
    report_type = 'Public Boards' if 'board' in filepath else 'MDAs'
    print(f"\n📄 Processing {report_type} {year}...")

    try:
        with pdfplumber.open(filepath) as pdf:
            for page_num, page in enumerate(pdf.pages):
                text = page.extract_text()
                if not text:
                    continue

                # Check if page contains irregularity keywords
                text_lower = text.lower()
                if any(kw in text_lower for kw in irregularity_keywords):
                    lines = text.split('\n')
                    for line in lines:
                        line_lower = line.lower()
                        if any(kw in line_lower for kw in irregularity_keywords):
                            flagged_entities.append({
                                'year': year,
                                'report_type': report_type,
                                'page': page_num + 1,
                                'extract': line.strip()[:200]
                            })
    except Exception as e:
        print(f"  ❌ Error: {e}")

ag_df = pd.DataFrame(flagged_entities)
print(f"\n✅ Done! Extracted {len(ag_df)} flagged lines across all 6 reports")
print(ag_df.head(10))

# Git: Cell 12 — extracted irregularity findings from 6 Auditor-General reports


📄 Processing Public Boards 2021...

📄 Processing MDAs 2021...

📄 Processing Public Boards 2022...

📄 Processing MDAs 2022...

📄 Processing Public Boards 2023...

📄 Processing MDAs 2023...

✅ Done! Extracted 2000 flagged lines across all 6 reports
   year    report_type  page  \
0  2021  Public Boards     5   
1  2021  Public Boards     9   
2  2021  Public Boards    12   
3  2021  Public Boards    12   
4  2021  Public Boards    12   
5  2021  Public Boards    12   
6  2021  Public Boards    12   
7  2021  Public Boards    12   
8  2021  Public Boards    12   
9  2021  Public Boards    12   

                                             extract  
0   17. Public Procurement Authority 898-906 189-191  
1  2000 (Act 584). This includes details of finan...  
2  5. Presented in table 1 is the financial impac...  
3  Table 1: Summary of financial irregularities f...  
4  NO Type of Irregularity % (GH¢) (US$) (€) (£) ...  
5  2 Cash Irregularities 2.89 485,765,755.61 3,05...  
6  3 Payroll I

In [12]:
# Cell 13 — Extract entity names from Auditor-General reports

# These are the entities already in our master dataset
known_entities = [
    'BOST', 'Ministry of Finance', 'National Petroleum Authority', 'NPA',
    'SSNIT', 'University of Ghana', 'Ghana Revenue Authority', 'GRA',
    'COCOBOD', 'Ghana Education Service', 'GES', 'NHIA', 'NHIS',
    'Electricity Company', 'ECG', 'Ghana Water', 'GWCL',
    'Bank of Ghana', 'BoG', 'GNPC', 'Ghana Ports', 'GPHA',
    'National Identification', 'NIA', 'Ghana Highway', 'Ghana Gas',
    'Tema Oil Refinery', 'TOR', 'VRA', 'Volta River',
    'Ghana Airports', 'GACL', 'Food and Drugs', 'FDA',
    'Forestry Commission', 'Lands Commission', 'GIPC',
    'Public Procurement Authority', 'PPA'
]

entity_flags = []

for filepath in ag_files:
    year = filepath.split('_')[-1].replace('.pdf', '')
    report_type = 'Public Boards' if 'board' in filepath else 'MDAs'
    print(f"\n📄 Scanning {report_type} {year}...")

    try:
        with pdfplumber.open(filepath) as pdf:
            for page_num, page in enumerate(pdf.pages):
                text = page.extract_text()
                if not text:
                    continue

                # Check if page mentions a known entity AND an irregularity
                text_lower = text.lower()
                has_irregularity = any(kw in text_lower for kw in irregularity_keywords)
                matched_entities = [e for e in known_entities if e.lower() in text_lower]

                if has_irregularity and matched_entities:
                    entity_flags.append({
                        'year': year,
                        'report_type': report_type,
                        'page': page_num + 1,
                        'entities_found': ', '.join(matched_entities),
                        'entity_count': len(matched_entities)
                    })

    except Exception as e:
        print(f"  ❌ Error: {e}")

entity_flag_df = pd.DataFrame(entity_flags)
print(f"\n✅ Done! Found {len(entity_flag_df)} pages flagging known entities")
print(entity_flag_df['entities_found'].value_counts().head(20))

# Git: Cell 13 — narrowed extraction to known entities from master dataset


📄 Scanning Public Boards 2021...

📄 Scanning MDAs 2021...

📄 Scanning Public Boards 2022...

📄 Scanning MDAs 2022...

📄 Scanning Public Boards 2023...

📄 Scanning MDAs 2023...

✅ Done! Found 633 pages flagging known entities
entities_found
TOR                                            175
GRA, TOR                                        70
Bank of Ghana, TOR                              39
GRA, Bank of Ghana, TOR                         20
GES, TOR                                        20
TOR, PPA                                        19
GRA, GES, TOR                                   18
TOR, Public Procurement Authority               16
Ghana Revenue Authority, GRA, TOR               10
NPA, TOR                                         8
Ministry of Finance, TOR                         8
ECG, TOR                                         8
Ghana Revenue Authority, TOR                     6
Ministry of Finance, Bank of Ghana, TOR          6
GRA, ECG, TOR                                

In [ ]:
# Cell 14 — Fix TOR dominance and get clean entity-level audit flags

# TOR appears to be in headers/footers — let's see what it actually is in context
# Also remove abbreviations that are too short and match too broadly
# We'll match full names only, then map to short names

entity_map = {
    'Bulk Oil Storage and Transportation': 'BOST',
    'Ministry of Finance': 'Ministry of Finance (MoF)',
    'National Petroleum Authority': 'National Petroleum Authority (NPA)',
    'Social Security and National Insurance Trust': 'SSNIT',
    'University of Ghana': 'University of Ghana (UG)',
    'Ghana Revenue Authority': 'Ghana Revenue Authority (GRA)',
    'Cocoa Board': 'COCOBOD',
    'Ghana Education Service': 'Ghana Education Service (GES)',
    'National Health Insurance': 'NHIA',
    'Electricity Company of Ghana': 'Electricity Company of Ghana (ECG)',
    'Ghana Water Company': 'GWCL',
    'Bank of Ghana': 'Bank of Ghana',
    'Ghana National Petroleum': 'GNPC',
    'Ghana Ports and Harbours': 'GPHA',
    'National Identification Authority': 'NIA',
    'Ghana Highways': 'Ghana Highway Authority',
    'Tema Oil Refinery': 'Tema Oil Refinery (TOR)',
    'Volta River Authority': 'VRA',
    'Ghana Airports': 'GACL',
    'Food and Drugs Authority': 'FDA',
    'Forestry Commission': 'Forestry Commission',
    'Lands Commission': 'Lands Commission',
}

clean_flags = []

for filepath in ag_files:
    year = filepath.split('_')[-1].replace('.pdf', '')
    report_type = 'Public Boards' if 'board' in filepath else 'MDAs'
    print(f"\n📄 Scanning {report_type} {year}...")

    try:
        with pdfplumber.open(filepath) as pdf:
            for page_num, page in enumerate(pdf.pages):
                text = page.extract_text()
                if not text:
                    continue

                text_lower = text.lower()
                has_irregularity = any(kw in text_lower for kw in irregularity_keywords)
                if not has_irregularity:
                    continue

                # Match full names only — no short abbreviations
                for full_name, standard_name in entity_map.items():
                    if full_name.lower() in text_lower:
                        clean_flags.append({
                            'year': year,
                            'report_type': report_type,
                            'page': page_num + 1,
                            'entity': standard_name,
                        })

    except Exception as e:
        print(f"  ❌ Error: {e}")

clean_flag_df = pd.DataFrame(clean_flags)
print(f"\n✅ Done! Found {len(clean_flag_df)} clean entity-level audit flags")
print(clean_flag_df.groupby(['entity', 'year']).size().reset_index(name='page_count').sort_values('page_count', ascending=False).head(25))

# Git: Cell 14 — fixed entity matching using full names, removed abbreviation noise


📄 Scanning Public Boards 2021...
